In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("../data/food_demand.csv")
features = [
    "day_of_week", "is_weekend", "exam_week", "special_event",
    "temperature_c", "menu_type", "expected_attendance"
]
X = df[features]
y = df["meals_consumed"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [ ]:
categorical_features = ["day_of_week", "menu_type"]
numerical_features = [
    "is_weekend", "exam_week", "special_event",
    "temperature_c", "expected_attendance"
]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", "passthrough", numerical_features)
])

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [ ]:
pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
y_pred = pipeline.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": np.round(y_pred).astype(int)
})
results.head(10)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f} meals")
print(f"RMSE : {rmse:.2f} meals")
print(f"R²   : {r2:.3f}")

In [ ]:
joblib.dump(pipeline, "../models/food_demand_model.joblib")
print("Saved model!")